# 로컬 문서 분석 및 PDF·HWPX 파싱

이미 받은 파일로 실습합니다. **다운로드 코드가 없습니다.** 아래 `INPUT_DIR`에 문서 폴더를 지정한 뒤 위에서부터 실행하세요. 프로젝트 `.venv` 커널을 사용합니다.

1. 파일 분석
2. 파싱 불가 파일 제외
3. PDF 파싱
4. HWPX 파싱
5. 문서별 문단 수·표 수·각 문단 첫 50글자 출력
6. 표 마크다운 변환 및 저장

기존 백엔드와 수업 노트북은 수정하지 않습니다. 제외란 처리 대상에서 빼는 것이며 원본을 삭제하지 않습니다.


In [1]:
from pathlib import Path
from collections import Counter
from dataclasses import asdict
from xml.etree import ElementTree as ET
import sys
import json
import re
import zipfile
import hashlib
import pdfplumber

HERE = Path.cwd().resolve()
ROOT = next((p for p in (HERE, *HERE.parents)
             if (p / "backend/app/rag/local_parsers.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("agent_practice 프로젝트 내부에서 실행하세요.")
if str(ROOT / "backend") not in sys.path:
    sys.path.insert(0, str(ROOT / "backend"))
from app.integrations.ports import ParsedBlock, ParsedDoc
from app.rag.local_parsers import parse_hwpx

DAY02 = ROOT / "sandbox/w5/day02"
INPUT_DIR = DAY02 / "public_documents"  # 새로 받은 파일 폴더로 변경하세요.
OUTPUT_DIR = DAY02 / "local_parsing_results"
if not INPUT_DIR.is_dir():
    raise FileNotFoundError(f"입력 폴더를 확인하세요: {INPUT_DIR}")
print("입력:", INPUT_DIR)
print("결과:", OUTPUT_DIR)

입력: /Users/pyoyoung-gyu/Desktop/Personal Project/한화아카데미/AI 서비스 백엔드 프로그래밍 실무/hanwha-agent/agent_practice/sandbox/w5/day02/public_documents
결과: /Users/pyoyoung-gyu/Desktop/Personal Project/한화아카데미/AI 서비스 백엔드 프로그래밍 실무/hanwha-agent/agent_practice/sandbox/w5/day02/local_parsing_results


## 1. 파일 분석
하위 폴더도 확인합니다. 숨김 파일과 결과 폴더는 제외하며 파일명·확장자·크기를 출력합니다.

In [2]:
files = sorted(p for p in INPUT_DIR.rglob("*")
               if p.is_file()
               and not any(part.startswith(".") for part in p.relative_to(INPUT_DIR).parts)
               and OUTPUT_DIR.resolve() not in p.resolve().parents)
if not files:
    raise ValueError("입력 폴더에 파일이 없습니다.")
for path in files:
    print(f"{path.relative_to(INPUT_DIR)} | {path.suffix.lower() or '(확장자 없음)'} | {path.stat().st_size:,} bytes")
print("\n총 파일 수:", len(files))
print("형식별 개수:", dict(Counter(p.suffix.lower() for p in files)))

01_mohw_committee_1.hwpx | .hwpx | 489,765 bytes
02_mohw_committee_2.hwpx | .hwpx | 486,882 bytes
03_mohw_committee_3.hwpx | .hwpx | 487,399 bytes
04_mohw_arpa_h.hwpx | .hwpx | 191,128 bytes
05_mohw_disability_education.hwpx | .hwpx | 506,361 bytes

총 파일 수: 5
형식별 개수: {'.hwpx': 5}


## 2. 파싱 불가한 파일 제외
이번 실습은 PDF와 HWPX만 지원합니다. `.hwp`, `.docx`, ZIP 등은 이번 처리 대상에서 제외합니다. HWPX 압축·본문 XML, PDF 열기·텍스트 유무를 검사합니다. 스캔 전용 PDF는 OCR이 필요하므로 제외합니다. 일부 페이지만 텍스트가 없는 PDF는 경고와 함께 텍스트가 있는 부분을 처리합니다.


In [3]:
candidates = {".pdf": [], ".hwpx": []}
excluded = []
warnings = []
parsed_docs: dict[Path, ParsedDoc] = {}
pdf_raw_tables = {}  # 6단계에서 마크다운으로 변환할 PDF 표 원본

def exclude(path, reason):
    excluded.append({"file": str(path.relative_to(INPUT_DIR)), "reason": reason})
    print("제외:", path.name, "→", reason)

for path in files:
    ext = path.suffix.lower()
    if ext not in candidates:
        exclude(path, "이번 실습의 지원 형식이 아님 (PDF/HWPX만 지원)")
        continue
    try:
        if path.stat().st_size == 0:
            raise ValueError("빈 파일")
        if ext == ".hwpx":
            with zipfile.ZipFile(path) as archive:
                sections = [n for n in archive.namelist()
                            if re.fullmatch(r"Contents/section\d+\.xml", n)]
                if not sections:
                    raise ValueError("HWPX 본문 XML 없음")
                for section in sections:
                    ET.fromstring(archive.read(section))
        else:
            with pdfplumber.open(path) as pdf:
                empty_pages = [i for i, page in enumerate(pdf.pages, 1)
                               if not (page.extract_text() or "").strip()]
                if len(empty_pages) == len(pdf.pages):
                    raise ValueError("추출 가능한 텍스트 없음 (스캔본은 OCR 필요)")
                if empty_pages:
                    message = f"{path.name}: 텍스트 없는 페이지 {empty_pages}; 해당 페이지는 OCR 확인 필요"
                    warnings.append(message)
                    print("주의:", message)
        candidates[ext].append(path)
    except Exception as exc:
        exclude(path, f"{type(exc).__name__}: {exc}")
print("\nPDF 후보:", len(candidates[".pdf"]), "HWPX 후보:", len(candidates[".hwpx"]))


PDF 후보: 0 HWPX 후보: 5


## 3. PDF 파싱
PDF는 원래 문단 구조가 보장되지 않습니다. 여기서는 **표 영역을 뺀 본문에서 줄 간격과 글머리표를 기준으로 묶은 추정 문단**을 셉니다. 표는 선 기반으로 탐지하므로 테두리 없는 표·복잡한 병합 셀은 누락될 수 있습니다. 다단 편집 문서는 읽기 순서를 별도로 확인하세요.

기존 백엔드 `parse_pdf()`는 페이지 단위 본문만 반환하므로 이 실습에서는 노트북 전용 함수를 사용합니다. 표 블록은 우선 행·열 데이터를 JSON 문자열로 보관하고 6단계에서 마크다운으로 바꿉니다.
참고: [pdfplumber 공식 문서](https://github.com/jsvine/pdfplumber)


In [4]:
def parse_pdf_assignment(path: Path):
    blocks = []
    raw_tables = {}
    table_count = 0
    with pdfplumber.open(path) as pdf:
        page_count = len(pdf.pages)
        for page_no, page in enumerate(pdf.pages, 1):
            events = []
            boxes = []
            for table in page.find_tables():
                rows = table.extract()
                if not rows or not any(str(cell or "").strip() for row in rows for cell in row):
                    continue
                table_count += 1
                locator = f"p.{page_no}/표{table_count}"
                raw_tables[locator] = rows
                boxes.append(table.bbox)
                events.append((table.bbox[1], table.bbox[0], ParsedBlock(
                    "표", locator, json.dumps(rows, ensure_ascii=False))))

            def outside_tables(obj):
                if obj.get("object_type") != "char":
                    return True
                x = (obj["x0"] + obj["x1"]) / 2
                y = (obj["top"] + obj["bottom"]) / 2
                return not any(x0 <= x <= x1 and top <= y <= bottom
                               for x0, top, x1, bottom in boxes)

            lines = page.filter(outside_tables).extract_text_lines(return_chars=False)
            groups = []
            group = []
            previous = None
            for line in lines:
                if not line["text"].strip():
                    continue
                new_paragraph = False
                if previous:
                    height = max(previous["bottom"] - previous["top"], 1)
                    gap = line["top"] - previous["bottom"]
                    crosses_table = any(previous["bottom"] <= box[1] < line["top"] for box in boxes)
                    new_paragraph = (gap > height * 0.65 or crosses_table
                                     or bool(re.match(r"^(?:[•●○□■ㅇ※]|\d+[.)])\s*", line["text"])))
                if new_paragraph and group:
                    groups.append(group)
                    group = []
                group.append(line)
                previous = line
            if group:
                groups.append(group)
            for i, group in enumerate(groups, 1):
                text = " ".join(line["text"].strip() for line in group)
                events.append((group[0]["top"], group[0]["x0"],
                               ParsedBlock("조항", f"p.{page_no}/문단{i}", text)))
            blocks.extend(block for _, _, block in sorted(events, key=lambda e: (e[0], e[1])))
    return ParsedDoc(blocks=blocks, page_count=page_count, table_count=table_count), raw_tables

for path in candidates[".pdf"]:
    try:
        doc, raw_tables = parse_pdf_assignment(path)
        if not doc.blocks:
            raise ValueError("본문과 표를 추출하지 못함")
        parsed_docs[path] = doc
        pdf_raw_tables[path] = raw_tables
        print("PDF 파싱 완료:", path.name)
    except Exception as exc:
        exclude(path, f"PDF 파싱 실패: {type(exc).__name__}: {exc}")
if not candidates[".pdf"]:
    print("PDF 파일이 없어 이 단계는 건너뜁니다.")

PDF 파일이 없어 이 단계는 건너뜁니다.


## 4. HWPX 파싱
기존 `backend/app/rag/local_parsers.py`의 `parse_hwpx()`를 호출합니다. 본문은 `조항`, 표는 `표` 블록이며 표는 기존 함수 안에서 이미 마크다운으로 변환됩니다. `page_count=1`은 기존 파서의 규약이며 실제 쪽수가 아닙니다. 복잡한 중첩 표·병합 셀은 원본과 비교가 필요합니다.


In [5]:
for path in candidates[".hwpx"]:
    try:
        doc = parse_hwpx(path)
        if not doc.blocks:
            raise ValueError("추출된 본문과 표가 없음")
        actual_tables = sum(b.kind == "표" for b in doc.blocks)
        if doc.table_count != actual_tables:
            warnings.append(f"{path.name}: 기존 파서의 표 수 {doc.table_count}와 실제 표 블록 수 {actual_tables}가 다릅니다.")
            doc.table_count = actual_tables
        parsed_docs[path] = doc
        print("HWPX 파싱 완료:", path.name)
    except Exception as exc:
        exclude(path, f"HWPX 파싱 실패: {type(exc).__name__}: {exc}")
if not candidates[".hwpx"]:
    print("HWPX 파일이 없어 이 단계는 건너뜁니다.")

HWPX 파싱 완료: 01_mohw_committee_1.hwpx
HWPX 파싱 완료: 02_mohw_committee_2.hwpx
HWPX 파싱 완료: 03_mohw_committee_3.hwpx
HWPX 파싱 완료: 04_mohw_arpa_h.hwpx
HWPX 파싱 완료: 05_mohw_disability_education.hwpx


## 5. 문단 수·표 수·각 문단 첫 50글자 출력
`parsed_docs`에 문서별 실제 `ParsedDoc` 객체가 담깁니다. 표 내부 텍스트는 본문 문단 수에서 제외합니다.

In [6]:
summary = []
for path, doc in parsed_docs.items():
    paragraphs = [b for b in doc.blocks if b.kind == "조항"]
    tables = [b for b in doc.blocks if b.kind == "표"]
    assert isinstance(doc, ParsedDoc)
    print("=" * 80)
    print(path.name)
    print(f"{type(doc).__name__} | 문단 {len(paragraphs)}개 | 표 {len(tables)}개")
    for i, block in enumerate(paragraphs, 1):
        print(f"문단 {i:03d} [{block.locator}]: {block.text[:50]}")
    summary.append({"file": str(path.relative_to(INPUT_DIR)),
                    "paragraph_count": len(paragraphs), "table_count": len(tables)})
print(f"\n성공 {len(parsed_docs)}개 / 제외 {len(excluded)}개")
for item in excluded:
    print("제외:", item["file"], "→", item["reason"])
for message in warnings:
    print("주의:", message)
if len(parsed_docs) < 5:
    print("과제 조건 미충족: 성공 문서가 최소 5개 필요합니다.")

01_mohw_committee_1.hwpx
ParsedDoc | 문단 19개 | 표 6개
문단 001 [01_mohw_committee_1]: 보건복지부(장관 조규홍)는 1월 23일(목) 2025년 제1차 첨단재생의료 및 첨단바이오의
문단 002 [01_mohw_committee_1]: 심의위원회는 화순전남대학교병원 등의 임상연구계획 총 2건(중위험 1건, 저위험 1건)을 심
문단 003 [01_mohw_committee_1]: 적합 의결된 과제는 진행성 췌장암* 환자를 대상으로 1차 화학요법으로써 mFOLFIRINO
문단 004 [01_mohw_committee_1]: * 췌장암은 5년 상대 생존율이 모든 암중에서 가장 낮고, 10년 암 관찰 생존율도 9.4
문단 005 [01_mohw_committee_1]: ** 췌장암을 치료하기 위한 항암요법
문단 006 [01_mohw_committee_1]: 표준항암요법인 폴리피녹스(FOLFIRINOX)는 4가지 항암제*를 복합처방하는 요법으로 많
문단 007 [01_mohw_committee_1]: * 옥살리플라틴, 이리노테칸, 류코보린, 5-플루오로우라실
문단 008 [01_mohw_committee_1]: 해당 연구는 FOLFIRINOX의 독성을 낮추기 위해 약제 용량을 감량한 mFOLFIRIN
문단 009 [01_mohw_committee_1]: 김우기 첨단재생의료 및 첨단바이오의약품 심의위원회 사무국장은 “심의위원회에서는 대체치료제가
문단 010 [01_mohw_committee_1]: “심의위원회는 재생의료기관에 연구계획 보완 기회를 부여하여 완성도를 높인 임상연구를 실시할
문단 011 [01_mohw_committee_1]: 사무국은 첨단재생의료 임상연구에 대한 소식과 정보를 전하기 위해 「첨단재생의료 임상연구소식
문단 012 [01_mohw_committee_1]: <붙임> 2025년 제1차 첨단재생의료 및 첨단바이오의약품 심의위원회 개요
문단 013 [01_mohw_committee_1]: <별첨>

## 6. 표 마크다운 변환 및 저장
PDF 표는 첫 행을 헤더로 사용하고 빈 셀·줄바꿈·파이프 문자를 처리하여 변환합니다. HWPX 표는 기존 파서에서 변환된 결과를 사용합니다. 문서별 `*_tables.md`, `*_parsed.json`, 전체 표 모음 `all_tables.md`, 요약과 제외 목록을 저장합니다. 실행마다 새 결과 폴더를 만들어 이전 결과와 섞이지 않게 합니다.


In [7]:
from datetime import datetime
import html

def table_to_markdown(rows):
    width = max((len(row) for row in rows), default=0)
    if not width:
        return ""
    def clean(value):
        return html.escape(str(value or "")).replace("|", "&#124;").replace("\r\n", "\n").replace("\n", "<br>")
    normalized = [[clean(cell) for cell in row] + [""] * (width - len(row)) for row in rows]
    lines = ["| " + " | ".join(normalized[0]) + " |", "| " + " | ".join(["---"] * width) + " |"]
    lines.extend("| " + " | ".join(row) + " |" for row in normalized[1:])
    return "\n".join(lines)

if not parsed_docs:
    raise ValueError("성공한 문서가 없습니다. 입력 파일과 제외 사유를 확인하세요.")
RUN_DIR = OUTPUT_DIR / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
RUN_DIR.mkdir(parents=True, exist_ok=False)
all_parts = ["# 파싱 문서의 표 모음"]
for path, doc in parsed_docs.items():
    if path in pdf_raw_tables:
        for block in doc.blocks:
            if block.kind == "표":
                block.text = table_to_markdown(pdf_raw_tables[path][block.locator])
    relative = path.relative_to(INPUT_DIR).as_posix()
    suffix = hashlib.sha256(relative.encode()).hexdigest()[:8]
    output_name = f"{path.stem}_{path.suffix[1:].lower()}_{suffix}"
    parts = [f"# {path.name}", f"원본: {relative}"]
    tables = [b for b in doc.blocks if b.kind == "표"]
    for i, block in enumerate(tables, 1):
        parts.extend([f"## 표 {i} ({block.locator})", block.text])
    if not tables:
        parts.append("추출된 표가 없습니다.")
    markdown = "\n\n".join(parts) + "\n"
    (RUN_DIR / f"{output_name}_tables.md").write_text(markdown, encoding="utf-8")
    (RUN_DIR / f"{output_name}_parsed.json").write_text(json.dumps(asdict(doc), ensure_ascii=False, indent=2), encoding="utf-8")
    all_parts.append(markdown)
(RUN_DIR / "all_tables.md").write_text("\n\n---\n\n".join(all_parts), encoding="utf-8")
for name, data in [("summary", summary), ("excluded", excluded), ("warnings", warnings)]:
    (RUN_DIR / f"{name}.json").write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
print("결과 폴더:", RUN_DIR)
print("성공 문서:", len(parsed_docs), "개")
print("전체 표:", sum(doc.table_count for doc in parsed_docs.values()), "개")
print("최소 5개 조건:", "충족" if len(parsed_docs) >= 5 else "미충족")

결과 폴더: /Users/pyoyoung-gyu/Desktop/Personal Project/한화아카데미/AI 서비스 백엔드 프로그래밍 실무/hanwha-agent/agent_practice/sandbox/w5/day02/local_parsing_results/20260929_160912_587721
성공 문서: 5 개
전체 표: 36 개
최소 5개 조건: 충족
